# Fix the first-half deck's charts

*Data Analysis in Python · Topic 8.5*

Last quarter's deck had two charts about the first half of 2025: revenue by month next to 2024's, and average order value by channel. One made the year look flat, the other made one channel look worthless. Measure what they hid, and redraw both honestly.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps), the cell that keeps January to June, the cell that adds the `channel_group` column (topic 4.2.'s steps) and the cell that draws the deck's charts. Look at the deck's charts, and read their axes.
2. In the first cell marked ✏️, create:
   - the `monthly_revenue` table: the `sessions` table's revenue totaled with one row per month (1 to 6) and one column per year (2024 and 2025)
   - `growth`: the first half of 2025's revenue compared with 2024's, as a fraction (0.12 for +12%)
   - `years_ax`: the Axes of one line chart with both years, one line each, a legend and a y axis that starts at 0
3. In the second cell marked ✏️, create:
   - `aov_by_channel`: each channel group's average order value in the first half of 2025, its revenue over its transactions, as a Series
   - `aov_ax`: the Axes of a horizontal bar chart of `aov_by_channel`, with the highest value on top and its value axis starting at 0
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `monthly_revenue` table has six months and two columns, 2024 and 2025, adding up to $827,836.35 and $1,215,370.78. `growth` is about `0.4681`. `years_ax` has two lines, one per year, a legend and a y axis starting at 0. `aov_by_channel` has nine channel groups, from Paid Social's $72.10 to Direct's $82.33. `aov_ax` has nine horizontal bars, sorted with Direct on top, its value axis starting at 0.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 7)]
START, END = "2024-01-01", "2025-06-30"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Only January to June: 2024's June file also has July 1 and 2 (lesson 3). Run it, and leave it as it is.
sessions = sessions[sessions["date"].dt.month <= 6].reset_index(drop=True)
sessions["date"].dt.year.value_counts()

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

In [ ]:
# The deck's two charts, as they were. Run it, and leave it as it is.
deck_fig, (deck_2024, deck_2025, deck_aov) = plt.subplots(1, 3, figsize=(16, 4))
for deck_ax, year in [(deck_2024, 2024), (deck_2025, 2025)]:
    year_rows = sessions[sessions["date"].dt.year == year]
    year_rows.groupby(year_rows["date"].dt.month)["revenue"].sum().plot(ax=deck_ax, marker="o", title=f"Revenue by month, H1 {year}")
first_half = sessions[sessions["date"].dt.year == 2025].groupby("channel_group")[["revenue", "transactions"]].sum()
(first_half["revenue"] / first_half["transactions"]).plot.bar(ax=deck_aov, title="Average order value, H1 2025")
deck_aov.set_ylim(71, 83)
deck_fig.tight_layout()
plt.show()

## Revenue by month

In [ ]:
# ✏️ Create monthly_revenue, growth and years_ax


## Average order value

In [ ]:
# ✏️ Create aov_by_channel and aov_ax


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib.axes
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def bars(ax):
        """Return the bars' values from bottom to top, or left to right, and whether they're horizontal."""
        patches = [patch for patch in ax.patches if patch.get_width() and patch.get_height()]
        if not patches:
            return [], None
        horizontal = len({round(patch.get_height(), 6) for patch in patches}) == 1 and len({round(patch.get_width(), 6) for patch in patches}) > 1
        if horizontal:
            ordered = sorted(patches, key=lambda patch: patch.get_y())
            ordered = ordered[::-1] if ax.yaxis_inverted() else ordered
            return [patch.get_width() for patch in ordered], True
        ordered = sorted(patches, key=lambda patch: patch.get_x())
        ordered = ordered[::-1] if ax.xaxis_inverted() else ordered
        return [patch.get_height() for patch in ordered], False

    TOTALS = [827836.35, 1215370.78]
    monthly = found.get("monthly_revenue")
    growth = found.get("growth")
    years_ax = found.get("years_ax")
    if not isinstance(monthly, pd.DataFrame):
        results.append((False, "monthly_revenue should be a DataFrame with the months as rows and the years as columns: sessions.pivot_table(index=sessions[\"date\"].dt.month, columns=sessions[\"date\"].dt.year, values=\"revenue\", aggfunc=\"sum\")."))
    elif monthly.shape == (2, 6):
        results.append((False, "The monthly_revenue table has the years as rows and the months as columns. Swap index and columns, so each year is a column and pandas draws it as a line."))
    elif monthly.shape != (6, 2) or [round(total, 2) for total in monthly.sum()] != TOTALS:
        results.append((False, f"The monthly_revenue table should have six months and two years, with columns adding up to $827,836.35 (2024) and $1,215,370.78 (2025). It is {monthly.shape[0]} × {monthly.shape[1]}. pivot_table() averages unless you pass aggfunc=\"sum\"."))
    elif not number(growth):
        results.append((False, f"growth should be a number: 2025's first-half revenue over 2024's, minus 1. It is {growth!r}."))
    elif abs(growth - 46.81) < 0.01:
        results.append((False, "growth is 46.81: that's a percentage. Keep it as a fraction, 0.4681, and format it when you show it."))
    elif abs(growth + 0.3189) < 0.0001:
        results.append((False, "growth is -0.3189: that's 2024 compared with 2025. Divide 2025 by 2024, then subtract 1."))
    elif abs(growth - 0.4681) > 0.0001:
        results.append((False, f"growth should be the first half of 2025's revenue over 2024's, minus 1: about 0.4681. It is {growth:.4f}."))
    elif not isinstance(years_ax, matplotlib.axes.Axes):
        results.append((False, f"years_ax should be the Axes of your line chart: fig, years_ax = plt.subplots(), then monthly_revenue.plot(ax=years_ax). It is a {type(years_ax).__name__}."))
    else:
        lines = years_ax.get_lines()
        sums = sorted(round(sum(line.get_ydata()), 2) for line in lines)
        if len(lines) == 1:
            results.append((False, "years_ax has one line. Put both years on the same Axes, so they share a scale: monthly_revenue.plot(ax=years_ax) draws one line per column."))
        elif len(lines) != 2 or sums != TOTALS:
            results.append((False, f"years_ax should have two lines, the two years' monthly revenue, adding up to $827,836.35 and $1,215,370.78. It has {len(lines)} lines."))
        elif years_ax.get_ylim()[0] != 0:
            results.append((False, f"years_ax's y axis starts at ${years_ax.get_ylim()[0]:,.0f}. The chart is about how much bigger 2025 was, so start it at 0: years_ax.set_ylim(bottom=0)."))
        elif years_ax.get_legend() is None:
            results.append((False, "years_ax has no legend, so the reader can't tell the years apart. Add one: years_ax.legend(), or plot the monthly_revenue table, which adds it."))
        else:
            results.append((True, f"Both years on one scale from 0: the first half of 2025 brought {growth:.1%} more revenue than 2024's, which the deck's two charts, each on its own scale, hid."))

    aov = found.get("aov_by_channel")
    aov_ax = found.get("aov_ax")
    if isinstance(aov, pd.DataFrame) and aov.shape[1] == 1:
        aov = aov.iloc[:, 0]
    if not isinstance(aov, pd.Series) or "Direct" not in aov.index:
        results.append((False, "aov_by_channel should be a Series with one value per channel group: total revenue and transactions by channel_group for 2025, then divide."))
    elif abs(aov["Direct"] - 97.23) < 0.01:
        results.append((False, "aov_by_channel's Direct is $97.23: that's the average of each row's revenue per transaction. Total revenue and transactions by channel group first, then divide (topic 4.4)."))
    elif abs(aov["Direct"] - 81.44) < 0.01:
        results.append((False, "aov_by_channel's Direct is $81.44: that's both years' first halves. Keep only 2025's rows first."))
    elif len(aov) != 9 or abs(aov["Direct"] - 82.33) > 0.01 or abs(aov["Paid Social"] - 72.10) > 0.01:
        results.append((False, f"aov_by_channel should have nine channel groups, from Paid Social's $72.10 to Direct's $82.33. It has {len(aov)}."))
    elif not isinstance(aov_ax, matplotlib.axes.Axes):
        results.append((False, f"aov_ax should be the Axes of your bar chart: fig, aov_ax = plt.subplots(), then aov_by_channel.sort_values().plot.barh(ax=aov_ax). It is a {type(aov_ax).__name__}."))
    else:
        values, horizontal = bars(aov_ax)
        if len(values) != 9 or abs(max(values) - aov.max()) > 0.01:
            results.append((False, f"aov_ax should have nine bars, aov_by_channel's values. It has {len(values)}."))
        elif horizontal is False:
            results.append((False, "aov_ax's bars are vertical. Make them horizontal, so the channel names fit: plot.barh(ax=aov_ax)."))
        elif values != sorted(values):
            results.append((False, "aov_ax's bars should be sorted with the highest value on top: aov_by_channel.sort_values().plot.barh(ax=aov_ax)."))
        elif aov_ax.get_xlim()[0] != 0:
            start = aov_ax.get_xlim()[0]
            results.append((False, f"aov_ax's value axis starts at ${start:,.2f}, so Direct's bar is {(aov.max() - start) / (aov.min() - start):.1f} times Paid Social's, for a difference of {aov.max() / aov.min() - 1:.0%}. Start it at 0: aov_ax.set_xlim(left=0)."))
        else:
            results.append((True, f"Average order value from ${aov.min():,.2f} to ${aov.max():,.2f}, on bars from 0: Direct's orders are {aov.max() / aov.min() - 1:.0%} bigger than Paid Social's, not the {(aov.max() - 71) / (aov.min() - 71):.0f} times the deck's chart showed."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
monthly_revenue = sessions.pivot_table(index=sessions["date"].dt.month, columns=sessions["date"].dt.year, values="revenue", aggfunc="sum")
totals = monthly_revenue.sum()
growth = totals[2025] / totals[2024] - 1
fig, years_ax = plt.subplots(figsize=(8, 4))
monthly_revenue.plot(ax=years_ax, marker="o", title="Revenue by month, first half of the year")
years_ax.set_ylim(bottom=0)
years_ax.yaxis.set_major_formatter("${x:,.0f}")
years_ax.set_xlabel("Month")
plt.show()
growth
```

```python
first_half = sessions[sessions["date"].dt.year == 2025].groupby("channel_group")[["revenue", "transactions"]].sum()
aov_by_channel = first_half["revenue"] / first_half["transactions"]
fig, aov_ax = plt.subplots(figsize=(8, 4))
aov_by_channel.sort_values().plot.barh(ax=aov_ax)
aov_ax.set_xlim(left=0)
aov_ax.xaxis.set_major_formatter("${x:,.0f}")
aov_ax.set_title("Average order value by channel group, H1 2025")
aov_ax.set_ylabel("")
plt.show()
```

</details>